# 03 — LOPO Results Analysis
Visualize Phase 2 LOPO cross-validation metrics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import matplotlib.pyplot as plt
import numpy as np

METRICS_PATH = '../outputs/lopo_metrics.json'

if not os.path.exists(METRICS_PATH):
    print(f'No metrics found at {METRICS_PATH}.')
    print('Run: python scripts/run_phase2_train.py --config config.yaml')
else:
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    print(f"Overall MRE: {metrics['mre_mean_mm']:.3f} ± {metrics['mre_std_mm']:.3f} mm")

In [ ]:
if 'metrics' in dir() and metrics:
    # Per-landmark MRE bar chart
    per_kp = metrics['per_landmark_mre']
    names = list(per_kp.keys())
    values = list(per_kp.values())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ['green' if v < 2.0 else 'orange' if v < 3.0 else 'red' for v in values]
    axes[0].barh(names, values, color=colors)
    axes[0].axvline(2.0, color='green', linestyle='--', alpha=0.7, label='2mm target')
    axes[0].axvline(3.0, color='orange', linestyle='--', alpha=0.7, label='3mm')
    axes[0].set_xlabel('MRE (mm)')
    axes[0].set_title('Per-Landmark MRE')
    axes[0].legend()

    # SDR bar chart
    sdr_keys = {k: v for k, v in metrics.items() if k.startswith('sdr_')}
    axes[1].bar([k.replace('sdr_', 'SDR@') for k in sdr_keys.keys()],
                [v * 100 for v in sdr_keys.values()], color='steelblue')
    axes[1].set_ylabel('Success Rate (%)')
    axes[1].set_ylim(0, 100)
    axes[1].axhline(80, color='green', linestyle='--', alpha=0.7, label='80% target')
    axes[1].set_title('Success Detection Rate')
    axes[1].legend()

    plt.tight_layout()
    plt.show()